In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 59.0 MB/s eta 0:00:00


In [ ]:
# Cell 1: Import libraries

import pandas as pd
import numpy as np
import random
import re

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import matplotlib.pyplot as plt

# Embeddings
from gensim.models import Word2Vec, FastText

In [ ]:
# Cell 2: Student behavior categories

motivated = [
    "I will complete my assignment today",
    "Let me start studying early",
    "I want to improve my grades",
    "Consistency is key to success",
    "I will practice coding daily"
]

procrastinator = [
    "I will do it tomorrow",
    "Let me watch one more episode",
    "I still have time",
    "I'll start after some time",
    "Deadline is far away"
]

stressed = [
    "Too many assignments to complete",
    "I am feeling overwhelmed",
    "Exams are giving me anxiety",
    "I don't think I can finish everything",
    "So much pressure from studies"
]

confident = [
    "I am well prepared for exams",
    "I can solve this easily",
    "I have practiced enough",
    "This subject is easy for me",
    "I am confident about my performance"
]

In [ ]:
# Cell 3: Create dataset

categories = {
    "motivated": motivated,
    "procrastinator": procrastinator,
    "stressed": stressed,
    "confident": confident
}

data = {"text": [], "label": []}

random.seed(101)
np.random.seed(101)

for _ in range(50):
    for label, phrases in categories.items():
        data["text"].append(random.choice(phrases))
        data["label"].append(label)

df = pd.DataFrame(data)

print("Dataset size:", df.shape)
df.head()

Dataset size: (200, 2)


,text,label
0,I will practice coding daily,motivated
1,Let me watch one more episode,procrastinator
2,So much pressure from studies,stressed
3,I have practiced enough,confident
4,Consistency is key to success,motivated


In [ ]:
# Cell 4: Add ambiguity and noise

overlap_phrases = [
    " but I am not sure",
    " maybe I will change my plan",
    " depends on the situation",
    " I am still thinking",
    " not fully confident though"
]

def add_noise(row):
    text = row["text"] + random.choice(overlap_phrases)

    if random.random() < 0.2:
        other_label = random.choice(list(categories.keys()))
        text += " however " + random.choice(categories[other_label]).lower()

    return text

df["text"] = df.apply(add_noise, axis=1)

df.head()

,text,label
0,I will practice coding daily but I am not sure,motivated
1,Let me watch one more episode depends on the s...,procrastinator
2,So much pressure from studies but I am not sure,stressed
3,I have practiced enough not fully confident th...,confident
4,Consistency is key to success depends on the s...,motivated


In [ ]:
# Cell 5: Clean + tokenize text

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    return text

def tokenize(text):
    return text.split()

df["clean_text"] = df["text"].apply(preprocess)
df["tokens"] = df["clean_text"].apply(tokenize)

df[["text", "tokens"]].head()

,text,tokens
0,I will practice coding daily but I am not sure,"[i, will, practice, coding, daily, but, i, am,..."
1,Let me watch one more episode depends on the s...,"[let, me, watch, one, more, episode, depends, ..."
2,So much pressure from studies but I am not sure,"[so, much, pressure, from, studies, but, i, am..."
3,I have practiced enough not fully confident th...,"[i, have, practiced, enough, not, fully, confi..."
4,Consistency is key to success depends on the s...,"[consistency, is, key, to, success, depends, o..."


In [ ]:
# Cell 6: Train Word2Vec

w2v_model = Word2Vec(
    sentences=df["tokens"],
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

print("Word2Vec model trained")

Word2Vec model trained


In [ ]:
# Cell 7: Sentence vector (Word2Vec)

def sentence_vector_w2v(tokens):
    vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

X = np.array([sentence_vector_w2v(tokens) for tokens in df["tokens"]])
y = df["label"]

print("Feature shape:", X.shape)

Feature shape: (200, 100)


In [ ]:
# Cell 8: Split dataset

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (160, 100)
Test: (40, 100)


In [ ]:
# Cell 9: Train models

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=200),
    "SVM": SVC(kernel="linear"),
    "Decision Tree": DecisionTreeClassifier()
}

results = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    results[name] = acc

    print(f"\n{name} Accuracy: {acc}")


Logistic Regression Accuracy: 0.175

SVM Accuracy: 0.175

Decision Tree Accuracy: 0.875


In [ ]:
# Cell 11: Train FastText

fasttext_model = FastText(
    sentences=df["tokens"],
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

print("FastText model trained")

FastText model trained


In [ ]:
# Cell 12: Convert sentence → FastText vector

def sentence_vector_fasttext(tokens):
    vectors = [fasttext_model.wv[word] for word in tokens]
    return np.mean(vectors, axis=0)

X_fasttext = np.array([sentence_vector_fasttext(tokens) for tokens in df["tokens"]])

In [ ]:
# Cell 13: Train-test split (FastText)

Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    X_fasttext, y, test_size=0.2, random_state=42
)

In [ ]:
# Cell 14: Train models (FastText)

fasttext_results = {}

for name, model in models.items():
    model.fit(Xf_train, yf_train)
    preds = model.predict(Xf_test)

    acc = accuracy_score(yf_test, preds)
    fasttext_results[name] = acc

    print(f"\n{name} (FastText) Accuracy: {acc}")


Logistic Regression (FastText) Accuracy: 0.175

SVM (FastText) Accuracy: 0.175

Decision Tree (FastText) Accuracy: 0.775


In [ ]:
# Cell 15: Compare embeddings

print("\nWord2Vec Results:", results)
print("FastText Results:", fasttext_results)


Word2Vec Results: {'Logistic Regression': 0.175, 'SVM': 0.175, 'Decision Tree': 0.875}
FastText Results: {'Logistic Regression': 0.175, 'SVM': 0.175, 'Decision Tree': 0.775}


In [ ]:
# Cell 16: Load GloVe embeddings

glove_path = "glove.6B.100d.txt"

glove_dict = {}

with open(glove_path, encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        glove_dict[word] = vector

print("GloVe loaded. Words:", len(glove_dict))

GloVe loaded. Words: 117048


In [ ]:
# Cell 17: Convert sentence → GloVe vector

def sentence_vector_glove(tokens):
    vectors = [glove_dict[word] for word in tokens if word in glove_dict]

    if len(vectors) == 0:
        return np.zeros(100)

    return np.mean(vectors, axis=0)

X_glove = np.array([sentence_vector_glove(tokens) for tokens in df["tokens"]])

In [ ]:
# Cell 18: Split dataset

Xg_train, Xg_test, yg_train, yg_test = train_test_split(
    X_glove, y, test_size=0.2, random_state=42
)

In [ ]:
# Cell 19: Train models (GloVe)

glove_results = {}

for name, model in models.items():
    model.fit(Xg_train, yg_train)
    preds = model.predict(Xg_test)

    acc = accuracy_score(yg_test, preds)
    glove_results[name] = acc

    print(f"\n{name} (GloVe) Accuracy: {acc}")


Logistic Regression (GloVe) Accuracy: 0.925

SVM (GloVe) Accuracy: 0.925

Decision Tree (GloVe) Accuracy: 0.75


In [ ]:
# Cell 20: Final comparison

print("\nWord2Vec Results:", results)
print("FastText Results:", fasttext_results)
print("GloVe Results:", glove_results)


Word2Vec Results: {'Logistic Regression': 0.175, 'SVM': 0.175, 'Decision Tree': 0.875}
FastText Results: {'Logistic Regression': 0.175, 'SVM': 0.175, 'Decision Tree': 0.775}
GloVe Results: {'Logistic Regression': 0.925, 'SVM': 0.925, 'Decision Tree': 0.75}


**Conclusion**

In [ ]:
# This project compared Word2Vec, FastText, and GloVe-style embeddings for student behavior classification. The results show that GloVe performed best with Logistic Regression and SVM, achieving the highest accuracy, while Word2Vec and FastText performed better with Decision Tree. This highlights that the choice of embedding and model significantly affects performance. Overall, GloVe provided better semantic representation for this dataset.